* 기본 설정

In [3]:
import os
import pathlib

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day04" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w3" / "day04"

print("프로젝트 루트  :", ROOT)

프로젝트 루트  : /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice


In [4]:
import anthropic
import sys

print("anthropic 버전 : ", anthropic.__version__)
print("지금의 파이썬 커널 버전 ", sys.executable)

anthropic 버전 :  1.6.0
지금의 파이썬 커널 버전  /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice/.venv/bin/python


In [8]:
import sys

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

In [9]:
from app.core.config import get_settings, mask

settings = get_settings()
print('모드        :', settings.app_mode)
print('모델        :', settings.llm_model)

if settings.anthropic_api_key is None:
    print('API 키      : (없음) — .env 의 ANTHROPIC_API_KEY 가 비어 있습니다')
else:
    print('API 키      :', mask(settings.anthropic_api_key.get_secret_value()))


모드        : mock
모델        : claude-haiku-4-5
API 키      : sk-ant-a...(108자)


In [ ]:
import inspect

from anthropic.resources.messages import Messages

# Message.creat : API 한테 보내줄 메시지 작성하는 기능
params = inspect.signature(Messages.create).parameters
# vlftn vkfkalxj whghl
required = [name for name, p in params.items() if p.default is inspect.Parameter.empty and name != "self"]
print(required)

QUESTION = "제주도 출장 숙박비 한도가 얼마인가요?"

try:
    Messages.create(
        None, # client가 와야하는 자리
        model="clauded-haiku-4-5",
        # max_tokens 입력 안하면 TypeError
        messages = [
            {"role" : "User", "content" : QUESTION}
        ],
    )
except TypeError as exc:
    print(type(exc).__name__)
    print(exc)

['max_tokens', 'messages', 'model']
TypeError
Missing required arguments; Expected either ('max_tokens', 'messages' and 'model') or ('max_tokens', 'messages', 'model' and 'stream') arguments to be given


* 금액 계산기

In [ ]:
PRICING = {'input': 1.0, 'cache_write': 1.25, 'cache_read': 0.1, 'output': 5.0}
USD_KRW = 1400.0

def cost_krw(input_tok: int, output_tok: int) -> float:
    usd = (input_tok * PRICING['input'] + output_tok * PRICING['output']) / 1000000
    return round(usd * USD_KRW, 1)

total_input = 69
output_tok = 146

(WED_INPUT, WED_OUTPUT) = (800, 400)
wed = cost_krw(WED_INPUT, WED_OUTPUT) # 가짜 금액
today = cost_krw(total_input, output_tok)
print(f'어제 어림한 값 : {wed } 원   (입력 {WED_INPUT} · 출력 {WED_OUTPUT} 토큰)')
print(f'오늘 실제 호출     : {today} 원   (입력 {total_input} · 출력 {output_tok} 토큰)')
print(f'차이               : {round(wed - today, 1)} 원')

어제 어림한 값 : 3.9 원   (입력 800 · 출력 400 토큰)
오늘 실제 호출     : 1.1 원   (입력 69 · 출력 146 토큰)
차이               : 2.8 원
